# 14v Verify 14e B1 From YAML
CPU-only verification that cached 14c Stage-2 TTA features reproduce the 14e B1 MTMC IDF1 target using `configs/datasets/cityflowv2.yaml` with no Stage-4 CLI overrides.

In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile

REPO_URL = "https://github.com/MRKDaGods/gp.git"
EXPECTED_MASTER_SHA_AT_BUILD = "327bf0512c4ea2fc7b336b1c49d0a40104c0b322"
WORK_DIR = Path("/kaggle/working")
PROJECT = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning master from {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", "master", REPO_URL, str(PROJECT)])
else:
    print("Repo already present; refreshing master ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "fetch", "origin", "master", "--depth", "1"])
    subprocess.check_call(["git", "-C", str(PROJECT), "checkout", "master"])
    subprocess.check_call(["git", "-C", str(PROJECT), "reset", "--hard", "origin/master"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
head_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("git rev-parse HEAD:", head_sha)
print("expected merged master SHA at notebook build:", EXPECTED_MASTER_SHA_AT_BUILD)

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip_install("faiss-cpu", "motmetrics", "loguru", "omegaconf", "rich", "networkx>=3.1", "click")
pip_install("filterpy", "ftfy", "lapx", "timm")
pip_install("--no-deps", "ultralytics")
pip_install("--no-deps", "boxmot==11.0.3")
pip_install("--no-deps", "-e", ".")

help_text = subprocess.check_output([sys.executable, "scripts/run_pipeline.py", "--help"], text=True)
if "--run-id" not in help_text:
    script_path = Path("scripts/run_pipeline.py")
    script_text = script_path.read_text(encoding="utf-8")
    script_text = script_text.replace(
        '@click.option("--dry-run", is_flag=True, default=False, help="Print resolved plan without running stages")\n',
        '@click.option("--dry-run", is_flag=True, default=False, help="Print resolved plan without running stages")\n'
        '@click.option("--run-id", default=None, help="Run id/name for the output directory")\n',
    )
    script_text = script_text.replace(
        'def main(config: str, dataset_config: str, stages: str, smoke_test: bool, dry_run: bool, override: tuple):',
        'def main(config: str, dataset_config: str, stages: str, smoke_test: bool, dry_run: bool, run_id: str | None, override: tuple):',
    )
    script_text = script_text.replace(
        '    if apply_cpu_when_no_cuda(cfg):\n',
        '    if run_id:\n        cfg.project.run_name = run_id\n\n    if apply_cpu_when_no_cuda(cfg):\n',
    )
    script_path.write_text(script_text, encoding="utf-8")
    print("Applied runtime-only --run-id compatibility shim to Kaggle clone")
else:
    print("run_pipeline.py already supports --run-id")

print("Setup complete")

In [ ]:
for mount in ["/tmp", "/kaggle/working"]:
    total, used, free = shutil.disk_usage(mount)
    print(f"{mount:16s} {free / 1024**3:.1f} GB free / {total / 1024**3:.1f} GB total")

candidate_mounts = [
    Path("/kaggle/input/data-aicity-2023-track-2"),
    Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
]
CITYFLOW_INPUT = next((path for path in candidate_mounts if path.exists()), None)
if CITYFLOW_INPUT is None:
    raise FileNotFoundError("CityFlowV2 dataset not found; attach thanhnguyenle/data-aicity-2023-track-2")
print(f"CityFlowV2 input: {CITYFLOW_INPUT}")

TMP_DATA = Path("/tmp/datasets")
TMP_DATA.mkdir(parents=True, exist_ok=True)
DATA_RAW_PARENT = PROJECT / "data" / "raw"
if not DATA_RAW_PARENT.is_symlink():
    if DATA_RAW_PARENT.exists():
        shutil.rmtree(DATA_RAW_PARENT)
    DATA_RAW_PARENT.parent.mkdir(parents=True, exist_ok=True)
    DATA_RAW_PARENT.symlink_to(TMP_DATA)

DATA_RAW = TMP_DATA / "cityflowv2"
DATA_RAW.mkdir(parents=True, exist_ok=True)
for split_dir in sorted(CITYFLOW_INPUT.iterdir()):
    if not split_dir.is_dir() or split_dir.name not in ("train", "validation", "test"):
        continue
    for scene_dir in sorted(split_dir.iterdir()):
        if not scene_dir.is_dir():
            continue
        for cam_dir in sorted(scene_dir.iterdir()):
            if not cam_dir.is_dir():
                continue
            flat_dir = DATA_RAW / f"{scene_dir.name}_{cam_dir.name}"
            if not flat_dir.exists():
                flat_dir.symlink_to(cam_dir)

cam_pattern = re.compile(r"^S\d{2}_c\d{3}$")
cams = sorted(path.name for path in DATA_RAW.iterdir() if path.is_dir() and cam_pattern.match(path.name))
print(f"CityFlowV2 ready: {len(cams)} cameras")
for cam in cams:
    print(f"  {cam}")

In [ ]:
INPUT_ROOT = Path("/kaggle/input")

def find_input_dir(slug: str, owner_slug: str, hints=()) -> Path:
    direct = INPUT_ROOT / slug
    if direct.exists():
        return direct
    owner, _, kernel = owner_slug.partition("/")
    nested = INPUT_ROOT / "notebooks" / owner / kernel
    if nested.exists():
        return nested
    lowered_slug = slug.lower()
    lowered_hints = tuple(str(hint).lower() for hint in hints)
    for path in list(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []:
        if not path.is_dir():
            continue
        name = path.name.lower()
        if lowered_slug in name or all(hint in name for hint in lowered_hints):
            return path
    for path in INPUT_ROOT.rglob("checkpoint.tar.gz") if INPUT_ROOT.exists() else []:
        parent_text = str(path.parent).lower()
        if lowered_slug in parent_text or all(hint in parent_text for hint in lowered_hints):
            return path.parent
    return direct

def find_kernel_output_file(owner_slug: str, filename: str, hints=()) -> Path:
    slug = owner_slug.split("/", 1)[1]
    input_dir = find_input_dir(slug, owner_slug, hints=hints)
    direct = input_dir / filename
    if direct.exists():
        print(f"Mounted {owner_slug}: {input_dir}")
        return direct
    if INPUT_ROOT.exists():
        for candidate in INPUT_ROOT.rglob(filename):
            parent_text = str(candidate.parent).lower()
            if slug.lower() in parent_text or all(str(hint).lower() in parent_text for hint in hints):
                print(f"Mounted {owner_slug}: {candidate.parent}")
                return candidate
    print(f"{filename} not found at {direct}; trying Kaggle API fallback for {owner_slug}")
    dl_dir = Path("/tmp") / f"kaggle_{slug}_download"
    dl_dir.mkdir(parents=True, exist_ok=True)
    pattern = "^" + filename.replace(".", r"\.") + "$"
    result = subprocess.run(["kaggle", "kernels", "output", owner_slug, "--file-pattern", pattern, "-p", str(dl_dir)], capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    downloaded = dl_dir / filename
    if downloaded.exists() and downloaded.stat().st_size > 0:
        print(f"Downloaded {filename} from {owner_slug}")
        return downloaded
    visible = [str(path) for path in INPUT_ROOT.rglob(filename)] if INPUT_ROOT.exists() else []
    raise FileNotFoundError(f"Could not resolve {filename} for {owner_slug}. Visible matches: {visible[:20]}")

def extract_checkpoint(checkpoint_path: Path, extract_dir: Path) -> tuple[str, Path]:
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {checkpoint_path} ({checkpoint_path.stat().st_size / 1024**2:.1f} MB)")
    with tarfile.open(str(checkpoint_path), "r:gz") as tar:
        tar.extractall(str(extract_dir))
    metadata_path = extract_dir / "run_metadata.json"
    if metadata_path.exists():
        previous_meta = json.loads(metadata_path.read_text(encoding="utf-8"))
        run_name = previous_meta["run_name"]
        return run_name, extract_dir / run_name
    run_dirs = [path for path in extract_dir.iterdir() if path.is_dir() and (path / "stage2").exists()]
    if len(run_dirs) == 1:
        return run_dirs[0].name, run_dirs[0]
    raise FileNotFoundError(f"No run_metadata.json or unique stage2 run dir found in {extract_dir}")

SOURCE_14C = "yahiaakhalafallah/14c-tta-stage2"
SOURCE_10A = "yahiaakhalafallah/mtmc-10a-stages-0-2"
checkpoint_14c = find_kernel_output_file(SOURCE_14C, "checkpoint.tar.gz", hints=("14c", "tta", "stage2"))
SOURCE_14C_RUN_NAME, SOURCE_14C_RUN_DIR = extract_checkpoint(checkpoint_14c, Path("/tmp/14c_checkpoint"))
print(f"Loaded 14c run: {SOURCE_14C_RUN_NAME}")
for stage in ["stage1", "stage2"]:
    stage_dir = SOURCE_14C_RUN_DIR / stage
    file_count = len([p for p in stage_dir.rglob("*") if p.is_file()]) if stage_dir.exists() else 0
    print(f"  14c {stage}: exists={stage_dir.exists()} files={file_count}")

SOURCE_10A_RUN_NAME = None
SOURCE_10A_RUN_DIR = None
if not (SOURCE_14C_RUN_DIR / "stage1").exists():
    checkpoint_10a = find_kernel_output_file(SOURCE_10A, "checkpoint.tar.gz", hints=("10a", "stages", "0", "2"))
    SOURCE_10A_RUN_NAME, SOURCE_10A_RUN_DIR = extract_checkpoint(checkpoint_10a, Path("/tmp/10a_checkpoint"))
    print(f"Loaded 10a fallback run: {SOURCE_10A_RUN_NAME}")

In [ ]:
RUN_ID = "run_verify"
OUTPUT_ROOT = PROJECT / "data" / "outputs"
RUN_DIR = OUTPUT_ROOT / RUN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if RUN_DIR.exists() or RUN_DIR.is_symlink():
    if RUN_DIR.is_symlink() or RUN_DIR.is_file():
        RUN_DIR.unlink()
    else:
        shutil.rmtree(RUN_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)

stage1_source = SOURCE_14C_RUN_DIR / "stage1"
if not stage1_source.exists() and SOURCE_10A_RUN_DIR is not None:
    stage1_source = SOURCE_10A_RUN_DIR / "stage1"
stage2_source = SOURCE_14C_RUN_DIR / "stage2"
if not stage1_source.exists():
    raise FileNotFoundError(f"Tracklet metadata stage1 not found in 14c or 10a fallback: {stage1_source}")
if not stage2_source.exists():
    raise FileNotFoundError(f"14c Stage 2 features not found: {stage2_source}")

for stage_name, source_dir in [("stage1", stage1_source), ("stage2", stage2_source)]:
    target = RUN_DIR / stage_name
    if target.exists() or target.is_symlink():
        if target.is_symlink() or target.is_file():
            target.unlink()
        else:
            shutil.rmtree(target)
    target.symlink_to(source_dir, target_is_directory=True)

run_latest = OUTPUT_ROOT / "run_latest"
if run_latest.exists() or run_latest.is_symlink():
    if run_latest.is_symlink() or run_latest.is_file():
        run_latest.unlink()
    else:
        shutil.rmtree(run_latest)
run_latest.symlink_to(RUN_DIR, target_is_directory=True)

required = [
    RUN_DIR / "stage1" / "tracklets.json",
    RUN_DIR / "stage2" / "embeddings.npy",
    RUN_DIR / "stage2" / "embeddings_tertiary.npy",
    RUN_DIR / "stage2" / "embedding_index.json",
    RUN_DIR / "stage2" / "hsv_features.npy",
]
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)
print(f"Linked Stage 1 from: {stage1_source}")
print(f"Linked Stage 2 from: {stage2_source}")
print(f"Run dir: {RUN_DIR}")
print(f"run_latest symlink: {run_latest} -> {run_latest.resolve()}")

In [ ]:
from omegaconf import OmegaConf
cfg = OmegaConf.load("configs/datasets/cityflowv2.yaml")
print("similarity_threshold:", cfg.stage4.association.graph.similarity_threshold)
print("aqe_k:", cfg.stage4.association.query_expansion.k)
print("fic_reg:", cfg.stage4.association.fic.regularisation)
print("w_tertiary:", cfg.stage4.association.tertiary_embeddings.weight)
print("v3_path:", cfg.stage2.reid.vehicle3.weights_path)
assert float(cfg.stage4.association.graph.similarity_threshold) == 0.48
assert int(cfg.stage4.association.query_expansion.k) == 2
assert float(cfg.stage4.association.fic.regularisation) == 0.5
assert float(cfg.stage4.association.tertiary_embeddings.weight) == 0.525
assert str(cfg.stage2.reid.vehicle3.weights_path).endswith("vehicle_transreid_dinov2_large_cityflowv2_final.pth")

In [ ]:
!python scripts/run_pipeline.py --config configs/datasets/cityflowv2.yaml --stages 3,4,5 --run-id run_verify

In [ ]:
import json
from pathlib import Path

metrics_path = Path("data/outputs/run_verify/stage5/evaluation_report.json")
if not metrics_path.exists():
    legacy_metrics_path = Path("data/outputs/run_verify/stage5/metrics.json")
    if legacy_metrics_path.exists():
        metrics_path = legacy_metrics_path
    else:
        raise FileNotFoundError(metrics_path)
metrics = json.load(open(metrics_path))
details = metrics.get("details", {}) or {}
idf1 = metrics.get("MTMC_IDF1") or metrics.get("mtmc_idf1") or details.get("mtmc_idf1") or metrics.get("idf1") or metrics["IDF1"]
id_switches = metrics.get("id_switches") or details.get("mtmc_id_switches") or metrics.get("IDS")
mtmc_id_switches = details.get("mtmc_id_switches")
TARGET_IDF1 = 0.77936
TARGET_ID_SWITCHES = 154
drift = float(idf1) - TARGET_IDF1
match = abs(drift) < 0.005 and int(id_switches) == TARGET_ID_SWITCHES
print(f"IDF1={float(idf1):.5f} (target {TARGET_IDF1}, drift {drift:+.5f})")
print(f"id_switches={int(id_switches)} (target {TARGET_ID_SWITCHES})")
print(f"mtmc_id_switches={mtmc_id_switches}")
print(f"MATCH={match}")
result = {
    "idf1": float(idf1),
    "id_switches": int(id_switches),
    "mtmc_id_switches": mtmc_id_switches,
    "target_idf1": TARGET_IDF1,
    "target_id_switches": TARGET_ID_SWITCHES,
    "drift": drift,
    "match": match,
    "metrics_path": str(metrics_path),
    "config_source": "configs/datasets/cityflowv2.yaml (no Stage-4 CLI overrides)",
}
with open("/kaggle/working/14v_verify_results.json", "w") as f:
    json.dump(result, f, indent=2)
assert abs(drift) < 0.005, f"DRIFT GATE FAILED: {drift}"
assert int(id_switches) == TARGET_ID_SWITCHES, f"ID SWITCH CHECK FAILED: {id_switches}"